# Private Multiplicative Weights for Histograms

This notebook builds a histogram from a truncated Gaussian mixture and uses MWEM to produce a differentially private approximation.

Before opening the notebook, run the following commands from the repository root:

```sh
julia --project=. -e 'using Pkg; Pkg.instantiate()'
julia -e 'using Pkg; Pkg.add(["IJulia", "UnicodePlots"])'
jupyter notebook examples/histograms.ipynb
```

In [ ]:
using PrivateMultiplicativeWeights
using Distributions
using UnicodePlots
using Random

Random.seed!(42)

## A mixture of two Gaussians

Start with a continuous mixture distribution truncated to the interval `[0, 1]`.

In [ ]:
distribution = truncated(
    MixtureModel(Normal, [(0.275, 0.075), (0.65, 0.1)], [0.65, 0.35]),
    0.0,
    1.0,
)

density_x = range(minimum(distribution), maximum(distribution), length=300)
density_y = pdf.(distribution, density_x)

lineplot(
    density_x,
    density_y;
    xlabel="value",
    ylabel="density",
    title="Truncated Gaussian mixture",
)

## Build an input histogram

Draw samples and count them in 100 equally sized bins. The binning clamps both endpoints, so values exactly equal to `0` or `1` remain valid.

In [ ]:
function histogram_from_sample(samples, num_bins)
    isempty(samples) && throw(ArgumentError("samples must not be empty"))
    0 <= minimum(samples) <= maximum(samples) <= 1 ||
        throw(ArgumentError("samples must lie in [0, 1]"))

    bins = clamp.(floor.(Int, samples .* num_bins) .+ 1, 1, num_bins)
    counts = zeros(Int, num_bins)
    for bin in bins
        counts[bin] += 1
    end

    collect(1:num_bins) ./ num_bins, counts
end

num_samples = 1_000
domain_size = 100
samples = rand(distribution, num_samples)
histogram_x, counts = histogram_from_sample(samples, domain_size)
@assert sum(counts) == num_samples

lineplot(
    histogram_x,
    counts;
    xlabel="value",
    ylabel="count",
    title="Sampled input histogram",
)

## Approximate the histogram with MWEM

`num_samples` is supplied explicitly because MWEM uses the source record count to calibrate noise.

In [ ]:
data = Histogram(counts, num_samples)
workload = SeriesRangeQueries(domain_size)
default_result = mwem(workload, data)
@assert isapprox(sum(default_result.synthetic.weights), 1.0)

default_plot = lineplot(
    histogram_x,
    counts;
    name="input",
    xlabel="value",
    ylabel="count",
    title="MWEM with default parameters",
)
lineplot!(
    default_plot,
    histogram_x,
    default_result.synthetic.weights .* num_samples;
    name="MWEM",
)

## Adjust the privacy and iteration parameters

In [ ]:
parameters = MWParameters(epsilon=0.1, iterations=8)
private_result = mwem(workload, data, parameters)
@assert isapprox(sum(private_result.synthetic.weights), 1.0)

private_plot = lineplot(
    histogram_x,
    counts;
    name="input",
    xlabel="value",
    ylabel="count",
    title="MWEM with epsilon=0.1 and 8 iterations",
)
lineplot!(
    private_plot,
    histogram_x,
    private_result.synthetic.weights .* num_samples;
    name="MWEM",
)

## Inspect approximation error

These error statistics are useful for evaluation, but they are **not** differentially private.

In [ ]:
(
    maximum_error=maximum_error(private_result),
    mean_squared_error=mean_squared_error(private_result),
)